In [1]:
import matplotlib.pyplot as plt
import datetime as dt
import pandas as pd
import numpy as np
import scipy as sp

from pathlib import Path
import soundfile as sf

In [2]:
import folium

In [3]:
import UBNA_localize_process_GCC__20250505 as localize

In [4]:
def plot_audio_seg_spectrogram(audio_features, spec_features):
    audio_seg = audio_features['audio_seg']
    fs = audio_features['sample_rate']
    start = audio_features['start']
    duration = audio_features['duration']

    vmax = spec_features['vmax']
    vmin = spec_features['vmin']
    cmap = spec_features['cmap']
    nfft = spec_features['NFFT']

    plt.figure(figsize=(15, 5))
    plt.rcParams.update({'font.size': 18})
    plt.title(f"Spectrogram", fontsize=24)
    plt.specgram(audio_seg+1e-6, NFFT=nfft, cmap=cmap, vmin=vmin, vmax=vmax, mode='magnitude', scale='dB')
    plt.yticks(ticks=np.linspace(0, 1, 6), labels=np.linspace(0, fs/2000, 6).astype('int'))
    plot_xtype = 'float'
    plt.xticks(ticks=np.linspace(0, duration*(fs/2), 11), 
            labels=np.round(np.linspace(start, start+duration, 11, dtype=plot_xtype), 2), rotation=30)
    plt.ylabel("Frequency (kHz)")
    plt.xlabel("Time (s)")
    plt.colorbar()
    plt.show()

In [5]:
SD_CARD_TO_AUDIOMOTH_NUM = {'STF_026': '014', 'STF_028' : '045', 
                            'STF_057': '015', 'STF_109': '008', 
                            'STF_114': '040', 'STF_053': '032', 
                            'STF_021': '016', 'STF_116': '036', 
                            'STF_060': '004', 'STF_025': '042',  
                            'STF_027': '020',
                            'STF_113': '044', 'STF_030': '012'}
FREQ_UPPER_LIM = 6/8
SD_CARD_TO_AUDIOMOTH_NUM

{'STF_026': '014',
 'STF_028': '045',
 'STF_057': '015',
 'STF_109': '008',
 'STF_114': '040',
 'STF_053': '032',
 'STF_021': '016',
 'STF_116': '036',
 'STF_060': '004',
 'STF_025': '042',
 'STF_027': '020',
 'STF_113': '044',
 'STF_030': '012'}

In [6]:
AUDIOMOTH_AT_PATCHA = {'014':'A1 (top)', '045':'A1 (bottom)',
                       '015':'A4 (top)', '008':'A4 (bottom)',
                       '040':'A7 (top)', '032':'A7 (bottom)'}
AUDIOMOTH_AT_PATCHE = {'016':'E2 (top)', '036':'E2 (bottom)',
                       '004':'E4 (top)', '042':'E4 (bottom)',
                       '020':'E8 (alone)',
                       '044':'E9 (top)', '012':'E9 (bottom)'}

AUDIOMOTH_AT_ALLPATCHES = AUDIOMOTH_AT_PATCHA | AUDIOMOTH_AT_PATCHE

In [7]:
audiomoth_to_color = {'014':'green', '045':'green',
                       '015':'red', '008':'red',
                       '040':'yellow', '032':'yellow',
                       '016':'blue', '036':'blue',
                       '004':'orange', '042':'orange',
                       '020':'pink',
                       '044':'brown', '012':'brown'}

In [8]:
from pyproj import Transformer

In [9]:
# WGS84 lat/lon/elevation -> Earth-centered Earth-fixed XYZ meters
lla_to_ecef = Transformer.from_crs(
    "EPSG:4979",  # lat, lon, height
    "EPSG:4978",  # geocentric X, Y, Z
    always_xy=True,
)

def distance_3d_m(lat1, lon1, elev1, lat2, lon2, elev2):
    x1, y1, z1 = lla_to_ecef.transform(lon1, lat1, elev1)
    x2, y2, z2 = lla_to_ecef.transform(lon2, lat2, elev2)

    return np.sqrt((x2 - x1)**2 + (y2 - y1)**2 + (z2 - z1)**2)

In [10]:
A1_ground_truth_latlon = [47 + (39.3437 / 60), -(122 + (17.8028 / 60))]
A4_ground_truth_latlon = [47 + (39.3243 / 60), -(122 + (17.8002 / 60))]
A7_ground_truth_latlon = [47 + (39.3061 / 60), -(122 + (17.8012 / 60))]

GROUND_TRUTH_LOCS = {'STF_026': A1_ground_truth_latlon, 'STF_028' : A1_ground_truth_latlon,
                     'STF_057': A4_ground_truth_latlon, 'STF_109': A4_ground_truth_latlon,
                     'STF_114': A7_ground_truth_latlon, 'STF_053': A7_ground_truth_latlon}

In [11]:
GROUND_TRUTH_LOCS

{'STF_026': [47.655728333333336, -122.29671333333333],
 'STF_028': [47.655728333333336, -122.29671333333333],
 'STF_057': [47.655405, -122.29667],
 'STF_109': [47.655405, -122.29667],
 'STF_114': [47.65510166666667, -122.29668666666667],
 'STF_053': [47.65510166666667, -122.29668666666667]}

In [12]:
lat1 = A7_ground_truth_latlon[0]
lon1 = A7_ground_truth_latlon[1]

for sd_card in list(SD_CARD_TO_AUDIOMOTH_NUM.keys()):
    if SD_CARD_TO_AUDIOMOTH_NUM[sd_card] in list(AUDIOMOTH_AT_PATCHA.keys()):
        lat2 = GROUND_TRUTH_LOCS[sd_card][0]
        lon2 = GROUND_TRUTH_LOCS[sd_card][1]
        # print(lat1, lon1, sd_card, lat2, lon2)

        distance_m = distance_3d_m(lat1, lon1, 2, lat2, lon2, 2)
        print(f"Ground-truth distance between STF_114 at {AUDIOMOTH_AT_PATCHA[SD_CARD_TO_AUDIOMOTH_NUM['STF_114']]} and {sd_card} at {AUDIOMOTH_AT_PATCHA[SD_CARD_TO_AUDIOMOTH_NUM[sd_card]]} is {distance_m}m")

Ground-truth distance between STF_114 at A7 (top) and STF_026 at A1 (top) is 69.70387923497152m
Ground-truth distance between STF_114 at A7 (top) and STF_028 at A1 (bottom) is 69.70387923497152m
Ground-truth distance between STF_114 at A7 (top) and STF_057 at A4 (top) is 33.748938210471806m
Ground-truth distance between STF_114 at A7 (top) and STF_109 at A4 (bottom) is 33.748938210471806m
Ground-truth distance between STF_114 at A7 (top) and STF_114 at A7 (top) is 0.0m
Ground-truth distance between STF_114 at A7 (top) and STF_053 at A7 (bottom) is 0.0m


In [13]:
from folium.plugins import MeasureControl

map = folium.Map(location=[47.65482429333334, -122.29547748000003], zoom_start=18, control_scale=True, max_zoom=24)

marker_locations = [("WJ's GPS @ A1", A1_ground_truth_latlon, "cyan"),
                    ("WJ's GPS @ A4", A4_ground_truth_latlon, "white"),
                    ("WJ's GPS @ A7", A7_ground_truth_latlon, "black"),
                    ("iPhone GPS point", [47.655096, -122.296703], "red")]

for name, location, fill_color in marker_locations:
    coordinate_label = f"{name}: {location[0]:.6f}, {location[1]:.6f}"
    folium.CircleMarker(location=location, radius=3, color="black", weight=1,
                        fill=True, fill_color=fill_color, fill_opacity=1,
                        tooltip=folium.Tooltip(coordinate_label, permanent=True,
                        direction="right", offset=(8, 0))).add_to(map)

# The ruler button measures a line drawn between any points on the map.
MeasureControl(position="topleft", primary_length_unit="meters", secondary_length_unit="feet").add_to(map)

map